# BattProDeep: dynamic validation and field application (open-source model)

This notebook demonstrates the **BattProDeep** framework: probabilistic aging predictions with trained neural networks for calendar and cyclic aging of LFP/graphite cells, applied incrementally to dynamic load profiles.

**Pipeline overview:**
1. Set up the environment and load the two laboratory validation profiles (PV-HESS and PV-FCR, Naumann et al.).
2. Inspect the C-rate and SOC of the profiles.
3. The stochastic looped engine and the validation run (`RUN_VALIDATION`).
4. Validation output: SOH trajectories with 95 % prediction interval.
5. Field application to home storage system 14 of Figgener et al. (`RUN_FIELD`), two bounding stress scenarios.
6. Field output: SOH trajectories and EOL distribution.

Both simulations are switched off by default (several hours to days each); the output sections load finished runs from `results/`.

In [ ]:
import os
import sys
from pathlib import Path
from importlib import reload
import gzip
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit

warnings.filterwarnings("ignore", category=FutureWarning)

# --- paths: the notebook lives in <repo>/notebooks, the package and data in <repo> ---
REPO_ROOT = Path(os.getcwd()).parent
for p in [str(REPO_ROOT), str(REPO_ROOT / "scripts")]:
    if p not in sys.path:
        sys.path.append(p)

from src.common.utils import calculate_soc_profile

# --- figure style (same as the paper figures) ---
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans"],
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "axes.linewidth": 1, "lines.linewidth": 1, "xtick.major.width": 0.9,
    "ytick.major.width": 0.9, "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

# --- 1. Validation profiles (Naumann et al.): PV home storage (PV-HESS) and PV + frequency containment reserve (PV-FCR) ---
PROFILE_DIR = REPO_ROOT / "dataset" / "data_dynamic_validation"
PROFILE_FILES = {"PV_Batterie": "PV_Batterie_Profil_Strom.txt",          # PV-HESS
                 "PV_PRL_Batterie": "PV_PRL_Batterie_Profil_Strom.txt"}  # PV-FCR
PROFILE_NAMES = {"PV_Batterie": "PV-HESS", "PV_PRL_Batterie": "PV-FCR"}
initial_soc = {"PV_Batterie": 28.025, "PV_PRL_Batterie": 60.025}        # % at the start of the laboratory test
repeat_dict = {"PV_Batterie": 1368, "PV_PRL_Batterie": 928}             # repetitions of the base profile in the test
temperature_dict = {"PV_Batterie": 40.8, "PV_PRL_Batterie": 40.6}       # degC, constant in the test
battery_capacity = 3.0                                                  # Ah, measured at the start of the test

# --- 2. Load the base current profiles (1-min resolution) ---
current_profiles, soc_profiles = {}, {}
for name, filename in PROFILE_FILES.items():
    current_profile = pd.read_csv(PROFILE_DIR / filename, delimiter="\t", header=None, usecols=[1])
    current_profiles[name] = current_profile
    soc_profiles[name] = calculate_soc_profile(current_profile, initial_soc=initial_soc[name],
                                               battery_capacity=battery_capacity)


def repeat_and_align_profiles(profiles_dict, repeat_dict, column_name, freq="min"):
    """Repeat each base profile as often as in the laboratory test and cut both to the shorter length."""
    repeated = {key: np.tile(np.asarray(profiles_dict[key]).reshape(-1), repeat_dict[key]) for key in profiles_dict}
    min_length = min(len(arr) for arr in repeated.values())
    index = pd.date_range(start=pd.Timestamp("2000-01-01"), periods=min_length, freq=freq)
    return {key: pd.DataFrame(arr[:min_length], index=index, columns=[column_name]) for key, arr in repeated.items()}


# --- 3. Input of the framework: Current (A), Temperature (degC), soc ---
# The framework only uses the FIRST value of "soc" (initial state); it integrates the SOC itself from the current.
current_profiles_df = repeat_and_align_profiles(current_profiles, repeat_dict, column_name="Current")
final_profiles = {}
for key, df in current_profiles_df.items():
    final_profiles[key] = df.assign(Temperature=temperature_dict[key], soc=initial_soc[key] / 100.0)
    print(f"{PROFILE_NAMES[key]}: {len(df) / 1440:.0f} days, base profile {len(current_profiles[key]) / 1440:.2f} days "
          f"repeated {repeat_dict[key]}x")


## Visualizing Validation Profiles
Before applying the aging models, we inspect the input sequences. The plot below illustrates the C-rate and the State of Charge (SOC) behavior over time for our two primary profiles: **PV-HESS** and **PV-FCR**.

In [ ]:
colors = ["#8FB6F5", "#17467F"]   # PV-HESS, PV-FCR (as in the paper)

fig, (ax_crate, ax_soc) = plt.subplots(2, 1, figsize=(7, 3.5), sharex=True)
for i, name in enumerate(PROFILE_FILES):
    ax_crate.plot(current_profiles[name].index, current_profiles[name][1] / battery_capacity,
                  label=PROFILE_NAMES[name], color=colors[i], alpha=0.8, lw=1)
    ax_soc.plot(soc_profiles[name].index, soc_profiles[name][1], label=PROFILE_NAMES[name], color=colors[i],
                alpha=0.8, lw=1)

ax_crate.set_ylabel("C-rate")
ax_crate.set_title("Validation profiles (one repetition of the base profile)")
ax_crate.legend(frameon=False)
ax_soc.set_ylabel("SOC")
ax_soc.set_xlabel("Time (minutes)")
for ax in (ax_crate, ax_soc):
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.7)
plt.tight_layout()
plt.show()


## The stochastic looped engine

Predicting battery degradation over years requires a path-dependent simulation. **BattProDeep** advances through the load profile in steps and carries the aged state from one step to the next.

### 1. Looped approach
The profile is processed in **3-month blocks**. The aged state of every trajectory at the end of a block, i.e. accumulated full equivalent cycles (FEC), virtual time ($vt$) and state of health (SOH), is the **initial condition** of the next block. The SOC is integrated from the current inside the framework, starting from the first value of the `soc` column.

### 2. Calendar and cyclic aging
Within each block, the stress factors are extracted by half-cycle counting and passed to probabilistic neural networks:

* **Cyclic aging:** FEC increment, depth of cycle, charge and discharge C-rates, temperature, SOC.
* **Calendar aging:** SOC, temperature and the virtual time $vt$, i.e. the storage time that would have produced the calendar loss already accumulated.

Both networks predict the mean ($\mu$) and standard deviation ($\sigma$) of the incremental capacity loss.

### 3. Trajectories and uncertainty
The simulation runs an ensemble of $M$ trajectories (here $M = 101$). Each trajectory draws its uncertainty values **once** and keeps them for the whole simulation, separately for calendar and cyclic aging (mode `sigma_e_reference_refsigma_independent`):

* **Aleatoric** ($\epsilon$): scales the predicted $\sigma$, i.e. the spread between replicate cells in the laboratory data.
* **Epistemic** ($\zeta$): selects a member of a 32-model bootstrap ensemble, i.e. the uncertainty of the model fit itself (`epistemic_support_scale = 1`, i.e. the full ensemble spread).

Trajectory 0 is the reference trajectory with $\epsilon = \zeta = 0$.

---

### Interpreting the band

*"Given the same usage profile, temperature history, and model uncertainty, these are plausible degradation outcomes."*

The plots show the mean of the trajectories and their **95 % prediction interval (PI)**:
* **lower edge (2.5 % quantile of SOH)** → pessimistic trajectory (fastest aging)
* **upper edge (97.5 % quantile of SOH)** → optimistic trajectory (slowest aging)

95 % of the model-consistent aging trajectories lie inside this band.


In [ ]:
from src.lfp.trajectory_uncertainty import run_outer_loop_variants_timebased_vectorized, load_particle_results


In [ ]:
# Validation run on the two laboratory profiles.
# Takes several hours per profile; the results are written to disk (disk-backed streaming).
RUN_VALIDATION = False          # True: run the simulation; False: section 4 loads the finished results
M_TRAJECTORIES = 101
EPISTEMIC_SUPPORT_SCALE = 1.0
VALIDATION_BASE = REPO_ROOT / "results" / "dynamic"
VALIDATION_DIR = VALIDATION_BASE / "pickles" / "9_fec_with_input_feature"   # descriptors + data folders
VALIDATION_RUN = f"validation_PV-HESS_PV-FCR_M{M_TRAJECTORIES}"            # name of the run index file

for name, df in final_profiles.items():   # sanity check of the framework input
    print(f"{PROFILE_NAMES[name]}: {df.index[0]} -> {df.index[-1]}, {len(df)} steps, "
          f"|I| max {df['Current'].abs().max():.2f} A, T {df['Temperature'].iloc[0]} degC, start SOC {df['soc'].iloc[0]:.3f}")

if RUN_VALIDATION:
    validation_results = run_outer_loop_variants_timebased_vectorized(
        variants=final_profiles,
        paths={"pickles_dir": str(VALIDATION_BASE / "pickles"), "run_base": str(VALIDATION_BASE)},
        C_nominal=battery_capacity,   # Ah
        fec_window=1,
        years=1,
        n_jobs=1,                     # profiles one after the other (memory)
        soc_max=1.0,
        soc_min=0.0,
        seed=42,
        M=M_TRAJECTORIES,
        n_boot=32,                    # members of the epistemic bootstrap ensemble
        bootstrap_seed=42,
        avoid_eps_zeta_tail_alignment=True,
        store_soc=False, store_fec=False,   # only calendar and cyclic loss are needed for the plots
        epistemic_mode="sigma_e_reference_refsigma_independent",
        epistemic_support_scale=EPISTEMIC_SUPPORT_SCALE,
        calendar_epistemic_model_folder=str(REPO_ROOT / "trained_models" / "calendar" / "probabilistic" / "02-epistemic"
                                            / "condition_bootstrap_models_20260527_103926"),
        cycle_epistemic_model_folder=str(REPO_ROOT / "trained_models" / "cycle" / "probabilistic" / "02-epistemic"
                                         / "trajectory_bootstrap_models_20260527_103926"),
        # aleatoric models: class default (trained_models/<calendar|cycle>/probabilistic/01-aleatoric)
        disk_backed_streaming=True,
        disk_backed_root=str(VALIDATION_DIR),
        results_index_name=VALIDATION_RUN,
    )
    print(validation_results)


## Evaluation: SOH trajectories on the validation profiles

`plot_soh_trajectories_twocol_streamed` is the plot function of the paper (Fig. 5). For every profile it shows
* the single SOH trajectories (thin lines),
* the **95 % prediction interval**: the envelope of the trajectories whose final SOH lies in the central 95 %,
* the mean of these trajectories.

SOH of a trajectory = 1 − cumulative (calendar + cyclic) loss. The arrays are read row by row from disk, so the
plot also works for long runs with limited memory.

In [ ]:
@njit
def compute_soh(cycle_loss, calendar_loss, n_steps):
    cum_loss = 0.0
    soh = np.empty(n_steps)
    for t in range(n_steps):
        cum_loss += cycle_loss[t] + calendar_loss[t]
        soh[t] = max(0.0, 1.0 - cum_loss)
    return soh


TRAJ_COLORS = ["#8FB6F5", "#17467F"]   # PV-HESS, PV-FCR (light / dark blue)
MEAN_COLORS = ["#B5670F", "#E08214"]   # orange means


def plot_soh_trajectories_twocol_streamed(particle_results_dict, max_days=None, ci_level=0.95,
                                          save_path=None, plot_points=1200, ylim=(0.855, 1.005)):
    """Paper Fig. 5: one panel per profile with trajectories, 95 % PI and mean."""
    n_profiles = len(particle_results_dict)
    fig, axs = plt.subplots(1, n_profiles, figsize=(7, 2), sharey=True)
    axs = np.atleast_1d(axs)
    q = (1.0 - ci_level) / 2.0

    for idx, (key, ax) in enumerate(zip(particle_results_dict.keys(), axs)):
        results = particle_results_dict[key]
        profile_name = PROFILE_NAMES.get(key, key)

        cycle_loss = results["cycle_loss"]
        calendar_loss = results["calendar_loss"]
        timestamps = results["timestamps"]

        n_particles = min(cycle_loss.shape[0], calendar_loss.shape[0])
        n_steps = min(cycle_loss.shape[1], calendar_loss.shape[1], len(timestamps))
        timestamps = np.asarray(timestamps[:n_steps])
        days = np.asarray((timestamps - timestamps[0]) / np.timedelta64(1, "D"), dtype=float)
        if max_days is not None:
            days = days[days <= max_days]
            n_steps = len(days)

        factor = max(1, n_steps // plot_points)
        plot_idx = np.arange(0, n_steps, factor, dtype=np.int64)
        if plot_idx[-1] != n_steps - 1:
            plot_idx = np.append(plot_idx, n_steps - 1)
        days_plot = days[plot_idx]

        # one trajectory at a time: only the thinned SOH matrix is kept in memory
        soh_matrix = np.empty((n_particles, len(plot_idx)), dtype=np.float64)
        for p in range(n_particles):
            soh_full = compute_soh(np.asarray(cycle_loss[p, :n_steps], dtype=np.float64),
                                   np.asarray(calendar_loss[p, :n_steps], dtype=np.float64), n_steps)
            soh_matrix[p] = soh_full[plot_idx]
            del soh_full

        ax.plot(days_plot, soh_matrix.T, color=TRAJ_COLORS[idx], alpha=0.15, lw=0.6, zorder=1)

        final_soh = soh_matrix[:, -1]
        low_final, high_final = np.quantile(final_soh, [q, 1.0 - q])
        selected = soh_matrix[(final_soh >= low_final) & (final_soh <= high_final)]
        lower, upper, mean = selected.min(axis=0), selected.max(axis=0), selected.mean(axis=0)
        print(f"{profile_name}: {len(selected)}/{n_particles} trajectories in the central {ci_level:.0%}, "
              f"final mean SOH {mean[-1]:.4f}, final PI {lower[-1]:.4f}-{upper[-1]:.4f}")

        ax.fill_between(days_plot, lower, upper, color=TRAJ_COLORS[idx], alpha=0.3, zorder=2,
                        label=f"{profile_name} {int(ci_level * 100)}% PI (BPD)")
        ax.plot(days_plot, mean, color=MEAN_COLORS[idx], lw=1, zorder=3, label=f"{profile_name} Mean (BPD)")

        ax.set_ylabel("State of Health (SOH)")
        ax.set_xlabel("Time (days)")
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.legend(frameon=False, ncol=1)

    plt.tight_layout()
    if save_path:
        plt.savefig(Path(save_path) / "soh_trajectories_val_wide.pdf", dpi=400, bbox_inches="tight",
                    pad_inches=0.005, transparent=False)
    plt.show()


## 4. Validation output

Every run writes an **index file** `<run name>_<timestamp>.pkl` that lists the result file of each profile. The cell
below loads the newest index file of `VALIDATION_RUN` (set in the run cell), so it always plots two profiles of the
**same** run, even if `VALIDATION_DIR` holds other runs. To plot a specific run, set `VALIDATION_INDEX` to its file name.

In [ ]:
VALIDATION_INDEX = None   # e.g. "validation_PV-HESS_PV-FCR_M101_20261111_093000.pkl"; None = newest of VALIDATION_RUN


def load_run_index(result_dir, run_name, index_file=None):
    """Load all profiles of one run via its index file. Result files are looked up next to the index
    file if the stored (absolute) path does not exist, e.g. after copying the results to another machine."""
    result_dir = Path(result_dir)
    if index_file is None:
        candidates = sorted(result_dir.glob(f"{run_name}_*.pkl"))
        if not candidates:
            raise FileNotFoundError(f"no run index '{run_name}_*.pkl' in {result_dir} - run the simulation first")
        index_path = candidates[-1]
    else:
        index_path = result_dir / index_file
    with open(index_path, "rb") as f:
        index = pickle.load(f)                      # {profile name: path of its result descriptor}
    loaded = {}
    for key, stored in index.items():
        path = Path(stored)
        if not path.exists():
            path = result_dir / Path(str(stored).replace("\\", "/")).name
        loaded[key] = load_particle_results(path)   # memory-mapped arrays; raises if the run is incomplete
        print(f"{key}: {path.name}, {loaded[key]['cycle_loss'].shape[0]} trajectories, "
              f"{loaded[key]['timestamps'][0]} -> {loaded[key]['timestamps'][-1]}")
    print("run index:", index_path.name)
    return loaded


try:
    validation_dict = load_run_index(VALIDATION_DIR, VALIDATION_RUN, VALIDATION_INDEX)
except FileNotFoundError as err:     # no run yet: report it and continue with the field part ("Run All" keeps going)
    validation_dict = None
    print(err)
if validation_dict:
    plot_soh_trajectories_twocol_streamed(validation_dict, ci_level=0.95, plot_points=1200, save_path=None)


## 5. Field application: home storage system 14 (Figgener et al.)

The trained framework is applied to the measured operation of a residential PV storage system (system 14 of the
dataset of Figgener et al.: 16s56p LFP cells, 4 modules of 16s14p). Two bounding scenarios are simulated at cell level:

* **Max stress:** current of the most loaded cell and temperature of the warmest cell (`I_in_A_cell_max`, `T_cell_max`)
* **Min stress:** current of the least loaded cell and temperature of the coldest cell (`I_in_A_cell_min`, `T_cell_min`)

The SOC of both scenarios is coulomb-counted from the **average** cell current (`Current_soc`), because parallel
cells share their SOC; the scenario current only sets the C-rates. The measured record (5.2 years) is extended to
the simulated duration by repeating its last whole years season-aligned (`repeat_profile_seasonal`).

**Input file:** `dataset/df_system_14_pub.csv` (index `Time`, 1-min steps, no gaps). Required columns: `I_in_A_cell_max`, `I_in_A_cell_min`, `T_cell_max`, `T_cell_min`, `I_in_A_corrected`. Optional: `SOC mit I-Verlust` (its first value sets the initial SOC, otherwise 0.5).

In [ ]:
import re

FIELD_DIR = REPO_ROOT / "results" / "field"


def _safe_module_name(s):
    """Sanitize a module / system name for folder creation."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")


def init_run_dirs(module_name):
    """Create results/field/<module>/{pickles,figures} and return the paths dict of the framework."""
    run_base = FIELD_DIR / _safe_module_name(module_name)
    pickles_dir, figures_dir = run_base / "pickles", run_base / "figures"
    pickles_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    return {"module": _safe_module_name(module_name), "run_base": str(run_base),
            "pickles_dir": str(pickles_dir), "figures_dir": str(figures_dir)}


def repeat_profile_seasonal(df, sim_years):
    """Extend the record to sim_years so every appended day sits in the same calendar season as its source.

    The full record is kept as is. After its end, the last whole years of the record are appended repeatedly.
    That window starts exactly n calendar years before the end of the record, so the first appended minute
    continues seamlessly with the month in which the record stops (e.g. September follows September).
    """
    dt = df.index[1] - df.index[0]
    end = df.index[-1]
    target_end = df.index[0] + pd.Timedelta(days=365.25 * sim_years)
    n_years = int((end + dt - df.index[0]) / pd.Timedelta(days=365.25))
    if n_years < 1:
        raise ValueError("record shorter than one year, cannot repeat it season-aligned")
    window = df.loc[end + dt - pd.DateOffset(years=n_years):]
    span = window.index[-1] - window.index[0] + dt          # n_years calendar years incl. leap days
    blocks, k = [df], 1
    while blocks[-1].index[-1] < target_end:
        blocks.append(window.set_axis(window.index + k * span))
        k += 1
    out = pd.concat(blocks).loc[:target_end]
    print(f"record {(end + dt - df.index[0]) / pd.Timedelta(days=365.25):.2f} a, repeating its last {n_years} a "
          f"({window.index[0]:%Y-%m-%d} -> {end:%Y-%m-%d}) {k - 1}x -> simulated "
          f"{(out.index[-1] - out.index[0] + dt) / pd.Timedelta(days=365.25):.2f} a")
    return out


In [ ]:
# --- field input: system 14 (public file, 1-min resolution) ---
# columns: I_in_A_cell_max / I_in_A_cell_min  cell current of the max / min stress scenario (A)
#          T_cell_max / T_cell_min            cell temperature of the max / min stress scenario (degC)
#          I_in_A_corrected                   system current (A); / 56 = average cell current -> Current_soc
#          SOC mit I-Verlust                  coulomb-counted SOC (0..1); only the first value is used (initial SOC)
FIELD_DATA = REPO_ROOT / "dataset" / "df_system_14_pub.csv"
N_CELLS_PARALLEL = 56          # 4 modules x 14 parallel cells
SIM_YEARS = 15.5               # simulated duration (measured record + season-aligned repetition)

REQUIRED = ["I_in_A_cell_max", "I_in_A_cell_min", "T_cell_max", "T_cell_min", "I_in_A_corrected"]
df_14 = pd.read_csv(FIELD_DATA, index_col="Time", parse_dates=True)
missing = [c for c in REQUIRED if c not in df_14.columns]
assert not missing, f"missing columns in {FIELD_DATA.name}: {missing}"
assert isinstance(df_14.index, pd.DatetimeIndex) and df_14.index.is_monotonic_increasing
assert df_14.index.to_series().diff().dropna().eq(pd.Timedelta(minutes=1)).all(), "index must be continuous 1-min"
assert not df_14[REQUIRED].isna().any().any(), "NaN in the input columns (the run stops on NaN in Current_soc)"
END_OF_MONITORING = df_14.index[-1]
print(f"system 14: {df_14.index[0]} -> {END_OF_MONITORING} "
      f"({(END_OF_MONITORING - df_14.index[0]).days / 365.25:.2f} a), {len(df_14):,} minutes")

df_rep = repeat_profile_seasonal(df_14[REQUIRED + (["SOC mit I-Verlust"] if "SOC mit I-Verlust" in df_14 else [])],
                                 SIM_YEARS)


In [ ]:
RUN_FIELD = False                # True: run both scenarios (about 2 days each); False: section 6 loads finished runs
M_FIELD = 101
FIELD_SUPPORT_SCALE = 1.0
FIELD_RUN = "sys14"              # prefix of the run index files
SCENARIOS = {"max_max_temp_signfix": ("I_in_A_cell_max", "T_cell_max"),
             "min_min_temp_signfix": ("I_in_A_cell_min", "T_cell_min")}
INITIAL_SOC = float(df_rep["SOC mit I-Verlust"].iloc[0]) if "SOC mit I-Verlust" in df_rep else 0.5

paths = init_run_dirs("figgener14")
FIELD_RESULT_DIR = Path(paths["pickles_dir"]) / "9_fec_with_input_feature"

if RUN_FIELD:
    import gc
    import inspect, src.lfp.trajectory_uncertainty as tu
    # an older trajectory_uncertainty.py ignores the Current_soc column silently (SOC from the scenario current)
    assert "Current_soc" in inspect.getsource(tu), "trajectory_uncertainty.py too old: it does not support Current_soc"
    for scenario, (cur_col, temp_col) in SCENARIOS.items():      # one after the other (memory)
        profile = pd.DataFrame({"Current": df_rep[cur_col], "Temperature": df_rep[temp_col],
                                "soc": INITIAL_SOC,       # only the first value is used (initial SOC)
                                "Current_soc": df_rep["I_in_A_corrected"] / N_CELLS_PARALLEL},  # average cell current
                               index=df_rep.index)
        print(f"--- starting {scenario} (M = {M_FIELD}) ---", flush=True)
        run_outer_loop_variants_timebased_vectorized(
            variants={scenario: profile},
            paths=paths, C_nominal=3.0, fec_window=1, years=1, n_jobs=1,
            soc_max=1, soc_min=0.05, seed=42, M=M_FIELD, n_boot=32, bootstrap_seed=42,
            avoid_eps_zeta_tail_alignment=True, store_soc=False, store_fec=False,
            epistemic_mode="sigma_e_reference_refsigma_independent",
            epistemic_support_scale=FIELD_SUPPORT_SCALE,
            calendar_epistemic_model_folder=str(REPO_ROOT / "trained_models" / "calendar" / "probabilistic" / "02-epistemic"
                                                / "condition_bootstrap_models_20260527_103926"),
            cycle_epistemic_model_folder=str(REPO_ROOT / "trained_models" / "cycle" / "probabilistic" / "02-epistemic"
                                             / "trajectory_bootstrap_models_20260527_103926"),
            # aleatoric models: class default (trained_models/<calendar|cycle>/probabilistic/01-aleatoric)
            disk_backed_streaming=True,
            disk_backed_root=str(FIELD_RESULT_DIR),
            results_index_name=f"{FIELD_RUN}_{scenario}_M{M_FIELD}",
        )
        print(f"--- finished {scenario} ---", flush=True)
        del profile
        gc.collect()


## 6. Field output: SOH trajectories and EOL distribution

In [ ]:
PROFILE_NAMES_FIG = ["Max Stress", "Min Stress"]
TRAJ_COLORS_FIG = ["#6A7D93", "#2B79F0"]     # grey-blue, medium blue
MEAN_COLORS_FIG = ["#A03F2E", "#005C00"]
FIELD_TEST_COLOR = "#3FA34D"


In [ ]:
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.image as mpimg
def plot_soh_and_eol_trajectories_figgener_4(
    particle_results_dict,
    SOH_EOL=0.9,
    max_days=None,
    measured_timestamps=None,
    measured_values=None,
    ci_level=0.95,
    bins=30,
    save_path=None,
    plot_points=1200,
    time_unit="days",       # "days" or "years"
    days_per_year=365.25,
    end_of_monitoring=None,  # timestamp of the last measured minute (vertical line)
):
    """SOH trajectories of both scenarios with their 95 % PI and the distribution of the EOL time.

    measured_values: field SOH values (fractions, already normalized) at measured_timestamps; optional.
    """
    assert time_unit in ("days", "years"), "time_unit must be 'days' or 'years'"

    def to_axis_unit(days_array):
        return days_array / days_per_year if time_unit == "years" else days_array

    fig = plt.figure(figsize=(7, 3.4))
    gs = fig.add_gridspec(2, 1, height_ratios=[2.2, 1.2], hspace=0.3)
    ax_top = fig.add_subplot(gs[0, 0])
    ax_hist = fig.add_subplot(gs[1, 0])

    q = (1.0 - ci_level) / 2.0
    eol_payloads = []

    for idx, key in enumerate(particle_results_dict.keys()):
        results = particle_results_dict[key]
        profile_name = PROFILE_NAMES_FIG[idx]

        cycle_loss = results["cycle_loss"]
        calendar_loss = results["calendar_loss"]
        timestamps = results["timestamps"]

        n_particles = min(cycle_loss.shape[0], calendar_loss.shape[0])
        n_steps = min(cycle_loss.shape[1], calendar_loss.shape[1], len(timestamps))
        timestamps = timestamps[:n_steps]

        if np.issubdtype(timestamps.dtype, np.datetime64):
            days = np.asarray((timestamps - timestamps[0]) / np.timedelta64(1, "D"), dtype=float)
        else:
            days = np.asarray(timestamps, dtype=float)

        if max_days is not None:
            days = days[days <= max_days]
            n_steps = len(days)

        if n_steps < 1:
            raise ValueError(f"No timesteps available for profile {key!r}.")

        factor = max(1, n_steps // plot_points)
        plot_idx = np.arange(0, n_steps, factor, dtype=np.int64)
        if plot_idx[-1] != n_steps - 1:
            plot_idx = np.append(plot_idx, n_steps - 1)

        days_plot = days[plot_idx]
        axis_plot = to_axis_unit(days_plot)

        soh_matrix = np.empty((n_particles, len(plot_idx)), dtype=np.float64)
        first_eol_days = np.full(n_particles, np.nan, dtype=np.float64)

        for p in range(n_particles):
            soh_full = compute_soh(cycle_loss[p, :n_steps], calendar_loss[p, :n_steps], n_steps)
            soh_matrix[p] = soh_full[plot_idx]

            eol_idx = np.argmax(soh_full <= SOH_EOL)
            if soh_full[eol_idx] <= SOH_EOL:
                first_eol_days[p] = days[eol_idx]

            del soh_full

        final_soh = soh_matrix[:, -1]
        final_low, final_high = np.quantile(final_soh, [q, 1.0 - q])
        trajectory_mask = (final_soh >= final_low) & (final_soh <= final_high)
        selected_soh = soh_matrix[trajectory_mask]

        lower = selected_soh.min(axis=0)
        upper = selected_soh.max(axis=0)
        mean = selected_soh.mean(axis=0)

        # prints always stay in days, regardless of time_unit
        print(f"{profile_name}: retained {trajectory_mask.sum()}/{n_particles} SOH trajectories")
        print(f"{profile_name} final mean SOH: {mean[-1]:.4f}")
        print(f"{profile_name} final SOH envelope: {lower[-1]:.4f}-{upper[-1]:.4f}")

        ax_top.plot(axis_plot, soh_matrix.T, color=TRAJ_COLORS_FIG[idx], alpha=0.15, lw=0.6, zorder=1)
        ax_top.fill_between(axis_plot, lower, upper, color=TRAJ_COLORS_FIG[idx], alpha=0.3, zorder=2,
                            label=f"{profile_name} {int(ci_level * 100)}% PI")
        ax_top.plot(axis_plot, mean, color=MEAN_COLORS_FIG[idx], lw=1.2, zorder=3, label=f"{profile_name} Mean")

        valid_eol = first_eol_days[np.isfinite(first_eol_days)]
        n_reached = len(valid_eol)
        n_not_reached = n_particles - n_reached

        if n_reached > 0:
            if n_not_reached:
                print(f"Warning: {profile_name}: {n_not_reached} trajectories did not reach EOL.")

            eol_low, eol_high = np.quantile(valid_eol, [q, 1.0 - q])
            selected_eol = valid_eol[(valid_eol >= eol_low) & (valid_eol <= eol_high)]

            # prints always stay in days
            print(f"{profile_name}: plotted {len(selected_eol)}/{n_reached} EOL dates")
            print(f"{profile_name} {ci_level:.0%} EOL range: {selected_eol.min():.1f}-{selected_eol.max():.1f} days")

            selected_eol_axis = to_axis_unit(selected_eol)
            eol_payloads.append({"name": profile_name, "values": selected_eol_axis, "color": TRAJ_COLORS_FIG[idx]})
        else:
            print(f"{profile_name}: no EOL crossings.")

    if eol_payloads:
        all_eol = np.concatenate([p["values"] for p in eol_payloads])
        eol_min, eol_max = all_eol.min(), all_eol.max()
        bin_edges = bins if eol_min == eol_max else np.linspace(eol_min, eol_max, bins + 1)

        for payload in eol_payloads:
            ax_hist.hist(payload["values"], bins=bin_edges, color=payload["color"], alpha=0.7, label="_nolegend_")

        ax_hist.axvline(eol_min, color=MEAN_COLORS_FIG[0], lw=1, linestyle="--", label="Earliest EOL")
        ax_hist.axvline(eol_max, color=MEAN_COLORS_FIG[1], lw=1, linestyle="--", label="Latest EOL")
        ax_hist.margins(x=0.05)
        if time_unit == "years":
            ax_hist.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    else:
        ax_hist.text(0.5, 0.5, "No EOL crossings", ha="center", va="center", transform=ax_hist.transAxes)

    ax_hist.set_xlabel(f"EOL expectance time ({time_unit}) for EOL threshold = {SOH_EOL*100:.0f}%", labelpad=1)
    ax_hist.set_ylabel("Count")
    ax_hist.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax_hist.legend(frameon=False, ncol=2)

    if measured_timestamps is not None and measured_values is not None:
        true_timestamps = pd.to_datetime(measured_timestamps, format="%d.%m.%Y %H:%M").to_numpy()
        true_soh = np.asarray(measured_values, dtype=float)          # SOH fractions
        reference_timestamps = next(iter(particle_results_dict.values()))["timestamps"]
        true_days = (true_timestamps - reference_timestamps[0]) / np.timedelta64(1, "D")

        if max_days is not None:
            measured_mask = true_days <= max_days
            true_days = true_days[measured_mask]
            true_soh = true_soh[measured_mask]

        true_axis = to_axis_unit(true_days)
        ax_top.scatter(true_axis, true_soh, color=FIELD_TEST_COLOR, s=15, marker="o", alpha=0.9,
                       zorder=7, label="SOH from field capacity tests")

    ax_top.axhline(SOH_EOL, color="#444444", linestyle=":", lw=1, label=f"EOL threshold ({SOH_EOL * 100:.0f}%)")

    reference_timestamps = next(iter(particle_results_dict.values()))["timestamps"]
    if end_of_monitoring is not None and np.issubdtype(reference_timestamps.dtype, np.datetime64):
        target_date = np.datetime64(pd.Timestamp(end_of_monitoring))
        target_day = (target_date - reference_timestamps[0]) / np.timedelta64(1, "D")
        target_axis = to_axis_unit(np.array([target_day]))[0]
        ax_top.axvline(target_axis, color="#444444", linestyle="-.", lw=1, label="End of Monitored Data")

    ax_top.set_ylabel("State of Health (SOH)")
    ax_top.set_xlabel(f"Time ({time_unit})", labelpad=1)
    ax_top.set_ylim(0.75, 1.005)

    sim_days = (np.asarray(reference_timestamps[-1]) - np.asarray(reference_timestamps[0])) / np.timedelta64(1, "D")
    max_days_xlim = max_days if max_days is not None else float(sim_days)
    xlim_axis = to_axis_unit(np.array([max_days_xlim]))[0]
    ax_top.set_xlim(0, xlim_axis)
    if time_unit == "years":
        ax_top.set_xticks(np.arange(0, np.ceil(xlim_axis) + 1, 2))

    ax_top.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax_top.legend(frameon=False, ncol=2, loc="upper right")

 

    if save_path:
        suffix = "" if time_unit == "days" else "_years"
        plt.savefig(Path(save_path) / f"soh_eol_common_histogram_reduced{suffix}.pdf", dpi=400, bbox_inches="tight",
                    pad_inches=0.005, transparent=False)
        
    

    plt.show()

In [ ]:
FIELD_INDEX = {"max_max_temp_signfix": None,   # e.g. "sys14_max_max_temp_signfix_M101_20261111_093000.pkl"
               "min_min_temp_signfix": None}   # None = newest index file of this scenario

field_dict = {}
for label, scenario in [("system_14_max", "max_max_temp_signfix"), ("system_14_min", "min_min_temp_signfix")]:
    try:
        run = load_run_index(FIELD_RESULT_DIR, f"{FIELD_RUN}_{scenario}_M{M_FIELD}", FIELD_INDEX[scenario])
    except FileNotFoundError as err:
        print(err)
        continue
    field_dict[label] = run[scenario]             # order = PROFILE_NAMES_FIG: Max Stress, Min Stress


In [ ]:
# Optional: field capacity tests as SOH (fractions) at their timestamps, e.g.
# FIELD_TEST_TIMES = ["02.06.2017 09:42", ...];  FIELD_TEST_SOH = [0.94, ...]
FIELD_TEST_TIMES = None
FIELD_TEST_SOH = None


In [ ]:
if len(field_dict) < 2:
    print("field results missing - run section 5 first (RUN_FIELD = True)")
else:
    plot_soh_and_eol_trajectories_figgener_4(
        particle_results_dict=field_dict,
        SOH_EOL=0.8,
        max_days=None,
        measured_timestamps=FIELD_TEST_TIMES,
        measured_values=FIELD_TEST_SOH,
        ci_level=0.95,
        bins=80,
        save_path=None,                     # e.g. paths["figures_dir"]
        plot_points=100000,
        time_unit="years",
        days_per_year=365.25,
        end_of_monitoring=END_OF_MONITORING,
    )
